# Activity 1: One Client, Three LLM Backends

**Week 6 Day 3 · OpenAI, Gemini, and Ollama through the same API**

Every large language model provider that matters today speaks a dialect of the same protocol: send a list of messages, get a message back. OpenAI defined the shape of that protocol first, and it caught on so widely that Google and the open-source Ollama project both accept it as-is. That means you can learn **one Python client** and point it at three completely different backends: a paid cloud API, a free-tier cloud API, and a model running on your own machine.

That is the entire idea of this notebook. Everything else is detail.

## What you will learn

- How to send a chat completion request and read the response
- Why the same prompt can produce a different answer twice, and how `temperature` controls that
- Why `stream=True` exists and what it changes
- That `base_url` is the only thing that changes between OpenAI, Gemini, and Ollama
- What `finish_reason` tells you about *why* the model stopped (you will need this in Activity 4)

## How to work through this

Run the cells in order, one at a time, and read the markdown between them before running the next. Several sections tell you **what to look for** in the output. Stop and compare what you actually got against that description before moving on. If they disagree, that is worth a question, not a silent shrug.

Two sections end in **Reflect** prompts. Answer them in a markdown cell in your own copy under `student-work/week6/day3/`. They have no answer key on purpose: they are the judgment calls the job actually consists of.

---
## Setup

One idea per cell from here on. Run them in order. This notebook assumes you completed [Activity 0](./Activity_0_Environment_and_API_Setup.md): `openai` and `python-dotenv` installed, `OPENAI_API_KEY` and `GOOGLE_API_KEY` in a repo-root `.env`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("OPENAI_API_KEY loaded:", "OPENAI_API_KEY" in os.environ)
print("GOOGLE_API_KEY loaded:", "GOOGLE_API_KEY" in os.environ)

---
# 1. Your first call

A chat completion request is a list of **messages**. Each message has a `role` (`system` sets behavior, `user` is the human turn, `assistant` is the model's turn) and `content` (the text). You send the whole conversation every time; the API has no memory of its own between calls.

We will reuse one example across this whole notebook: an insurance claim note that needs summarizing for a supervisor. Keep an eye on how the *code* barely changes as the *backend* changes underneath it.

In [ ]:
CLAIM_NOTE = (
    "Insured reports rear-end collision at low speed in a parking lot. "
    "Bumper cover cracked, no airbag deployment, other party's insurance "
    "already confirmed liability. Insured requests expedited repair "
    "authorization due to upcoming work travel."
)

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a claims operations assistant."},
        {"role": "user", "content": f"Summarize this claim note in one sentence for a supervisor:\n\n{CLAIM_NOTE}"},
    ],
)

print(completion.choices[0].message.content)

`completion` is not just the text you printed. It is a typed object with everything the API returned: which model actually answered, how many tokens the prompt and the reply cost, and *why* the model stopped generating. That last field, `finish_reason`, seems minor now. In Activity 4 it is the whole mechanism that tells you a model wants to call a function instead of replying in plain text.

In [ ]:
print("model:", completion.model)
print("finish_reason:", completion.choices[0].finish_reason)
print("prompt tokens:", completion.usage.prompt_tokens)
print("completion tokens:", completion.usage.completion_tokens)

`finish_reason` is `"stop"` here: the model produced a complete answer and stopped on its own. Other values exist, `"length"` means it hit `max_tokens` mid-thought, and `"tool_calls"` means it stopped on purpose because it wants you to run a function before it continues. Remember that third one.

---
# 2. Same prompt, different answer

Go back and run the section 1 cell a second time. You will often get a summary worded differently from the first, even though not one character of your code changed.

That is not a bug, and understanding why is the most important idea in this notebook.

A model does not look up an answer. At every single step it produces a **probability distribution** over its entire vocabulary: a ranked list of how likely each possible next token is. Something then has to *pick* one token off that list. That picking step is called **sampling**, and by default it is random, weighted by those probabilities.

So the model never decides "the answer is X." It decides something closer to "there is a 31 percent chance the next token is `The`, a 12 percent chance it is `Insured`, a 4 percent chance it is `Low`," and then rolls a weighted die.

Let us watch the die roll. Run this cell and read the five answers.

In [ ]:
QUESTION = (
    "Name one thing a claims adjuster inspects after a parking lot collision. "
    "Reply with the noun phrase only, no punctuation."
)


def ask_once(temperature=None):
    kwargs = {"model": "gpt-4o-mini", "messages": [{"role": "user", "content": QUESTION}]}
    if temperature is not None:
        kwargs["temperature"] = temperature
    return client.chat.completions.create(**kwargs).choices[0].message.content.strip()


for i in range(5):
    print(i + 1, ask_once())

**What to look for:** you may have gotten five identical answers, or three of one and two of another, or five different ones. All of those are normal, and *which* you got is itself informative. If the model's probability distribution was sharply peaked on one obvious answer, sampling picks that answer nearly every time. If several answers were close in probability, you see variety.

If you got five identical lines, do not assume the demo failed. Run the cell again, or make the question more open-ended, and the variety will show up.

So the randomness is real, but you can control how much of it you get. The dial is a parameter called **temperature**.

Temperature reshapes the probability distribution *before* a token is sampled:

- **Low temperature** sharpens the distribution. The already-likely tokens become even more likely, and the model almost always takes its top choice.
- **High temperature** flattens the distribution. Unlikely tokens get a real chance, and output gets more varied and eventually incoherent.

Set it to `0` and the model takes its highest-probability token every time. Watch what that does.

In [ ]:
for i in range(5):
    print(i + 1, ask_once(temperature=0))

**What to look for:** at `temperature=0` you should see the same answer five times, or very close to it.

One honest caveat, because you will eventually hit it and wonder if you broke something: `temperature=0` makes sampling *greedy*, meaning "always take the highest-probability token." It does not make the API a hard guarantee of identical output. Providers batch requests across many users, and floating point arithmetic on a GPU is not perfectly associative, so the ranking of two nearly-tied tokens can flip between runs. In practice `temperature=0` is highly repeatable and is what you should use for pipeline work. Treat it as "as deterministic as this system gets," not as a promise.

Now push it the other way. Temperature accepts values from `0` to `2`. High temperature flattens the distribution, which means unlikely tokens get a real chance of being picked.

In [ ]:
for i in range(5):
    print(i + 1, ask_once(temperature=1.6))

## Choosing a temperature is a judgment call, not a default

There is no "correct" temperature. There is only a temperature that fits the job:

| You are building | Temperature | Why |
|---|---|---|
| Extracting fields from claim notes into a table | `0` | The same note must produce the same row every time, or your pipeline is not reproducible |
| Classifying a message as urgent or routine | `0` | You want the model's single best judgment, not a sample of its opinions |
| Drafting customer-facing copy a human will edit | `0.7` to `1.0` | Some variety is useful, a human reviews it anyway |
| Brainstorming ten different names for a project | `1.2`+ | Variety *is* the product |

Most data engineering work sits in the first two rows. That is why `temperature=0` shows up in Activity 5's ReAct agent: an agent deciding which tool to call next should not roll dice on that decision.

### Reflect before moving on

Write your answers in a markdown cell in your own copy. There is no answer key, these are judgment questions.

1. Your team ships a pipeline that summarizes 10,000 claim notes nightly. A regulator later asks you to prove that a specific note produced a specific summary on a specific date. What temperature did you need to have used, and what else would you need to have stored besides the temperature?
2. A colleague says "temperature 0 makes the model accurate." Based on what you just saw, what is wrong with that sentence? Be precise about the difference between *consistent* and *correct*.
3. Look back at the streaming section you are about to read. If a model is sampling one token at a time, what does that tell you about whether it "knows" how its sentence will end when it emits the first word?

---
# 3. Streaming: watch the answer arrive

A model does not generate a full paragraph and then hand it to you. It generates one token at a time, sampling each one from a fresh probability distribution exactly as you just saw. `chat.completions.create` waits for every token before returning, so a long answer feels slow even though the model started responding immediately. `stream=True` gives you each token as it is produced instead of waiting for all of them.

In [ ]:
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain what 'liability' means in an auto insurance claim, in two sentences."}
    ],
    stream=True,
)

for chunk in stream:
    piece = chunk.choices[0].delta.content
    if piece:
        print(piece, end="", flush=True)

Each `chunk` is a small slice of the response. Most chunks carry a piece of text in `delta.content`; the very last chunk carries no text at all, just the stop signal, which is why the `if piece:` check is there. This is the same pattern a chat UI uses to render text as it types.

You can render the growing response as live Markdown instead of raw `print` by updating a display handle in place.

In [ ]:
from IPython.display import display, Markdown

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "List, as a short Markdown table, three things a claims adjuster checks before approving a payout."}
    ],
    stream=True,
)

full_text = ""
handle = display(Markdown(""), display_id=True)
for chunk in stream:
    piece = chunk.choices[0].delta.content
    if piece:
        full_text += piece
        handle.update(Markdown(full_text))

---
# 4. One client, many backends

Here is the part that makes this notebook worth doing. `OpenAI(...)` takes two arguments that matter: `api_key`, and `base_url`, the address it sends requests to. Change `base_url` and `api_key`, and the exact same `client.chat.completions.create(...)` call now talks to a different provider. Nothing else in your code changes.

| Provider | Runs where | Cost | `base_url` |
|---|---|---|---|
| **OpenAI** | Cloud | Pay per token | *(default, `api.openai.com`)* |
| **Gemini** | Cloud | Free tier (AI Studio) | `generativelanguage.googleapis.com/v1beta/openai/` |
| **Ollama** | Your machine | Free (your hardware) | `http://localhost:11434/v1` |

## 4a. Gemini, through the OpenAI client

This is not the `google-genai` SDK you may see elsewhere in this course. It is the plain `openai` package again, pointed at Google's OpenAI-compatible endpoint. Same `.chat.completions.create(...)`, same message format, same `stream=True`, and `temperature` means the same thing here too.

In [ ]:
gemini_client = OpenAI(
    api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

completion = gemini_client.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=[
        {"role": "system", "content": "You are a claims operations assistant."},
        {"role": "user", "content": f"Summarize this claim note in one sentence for a supervisor:\n\n{CLAIM_NOTE}"},
    ],
)

print(completion.choices[0].message.content)

Same prompt, same code shape, different model answering. Streaming works exactly the same way too, no new syntax to learn.

In [ ]:
stream = gemini_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[{"role": "user", "content": "Explain what 'liability' means in an auto insurance claim, in two sentences."}],
    stream=True,
)

for chunk in stream:
    piece = chunk.choices[0].delta.content
    if piece:
        print(piece, end="", flush=True)

## 4b. Ollama, running on your own machine

[Ollama](https://ollama.com/) runs open models locally and, like Gemini, speaks the OpenAI-compatible protocol on its own server. This part is optional: it needs Ollama installed and a model pulled, which takes a few minutes and some disk space. If you skip it, you still have the core lesson from 4a.

Run these two commands in a **terminal**, not a notebook cell, since the server needs to keep running in the background:

```bash
# 1. Install Ollama (Linux)
curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the server, then in a second terminal, pull a small model
ollama serve
ollama pull llama3.2:1b
```

`llama3.2:1b` is small enough to run on a laptop or the classroom VM. If disk or RAM is tight, `qwen2.5:0.5b` is even smaller.

In [ ]:
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # required by the client, ignored by Ollama
)

try:
    completion = ollama_client.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": "You are a claims operations assistant."},
            {"role": "user", "content": f"Summarize this claim note in one sentence for a supervisor:\n\n{CLAIM_NOTE}"},
        ],
    )
    print(completion.choices[0].message.content)
except Exception as e:
    print("Ollama not reachable, is 'ollama serve' running in a terminal?")
    print(f"  ({e})")

---
# Checkpoint: what actually changed

Look back across sections 1 through 4. The `messages` list never changed shape. The `.chat.completions.create(...)` call never changed shape. `stream=True` behaved identically everywhere, and so did `temperature`. The only lines that changed between providers were the two arguments to `OpenAI(...)`: `api_key` and `base_url`.

That is the whole idea behind API compatibility standards: a stable client interface over swappable backends. Production systems use exactly this pattern to avoid locking themselves into one vendor.

---
# Your Turn

Work in your own copy under `student-work/week6/day3/`.

Write a function `compare_providers(prompt, clients)` that takes a prompt string and a dictionary of `{provider_name: (client, model_name)}`, calls each one with the same prompt, and prints the provider name next to its answer. Run it on the claim note summarization prompt using at least two of the three clients you built above (Ollama is optional).

```python
clients = {
    "openai": (client, "gpt-4o-mini"),
    "gemini": (gemini_client, "gemini-2.5-flash"),
    # "ollama": (ollama_client, "llama3.2:1b"),
}

def compare_providers(prompt, clients):
    # TODO: loop over clients.items(), call chat.completions.create for each,
    # print the provider name and the response content
    ...

compare_providers(f"Summarize this claim note in one sentence:\n\n{CLAIM_NOTE}", clients)
```

**Stretch goal:** time each call with `time.perf_counter()` and print latency next to each answer. Which backend is fastest for this prompt? Does that match what you would expect from a cloud API versus a local model?

## What you did

- Sent chat completion requests and read the full response object, not just the text.
- Saw that a model samples each token from a probability distribution, and used `temperature` to control how much randomness that sampling gets.
- Learned why `temperature=0` is the right default for pipeline work, and why it is "as repeatable as this system gets" rather than a guarantee.
- Used `stream=True` to render output as it is generated.
- Learned that `finish_reason` tells you *why* a model stopped, not just *that* it stopped.
- Pointed the same `openai` client at OpenAI, Gemini, and (optionally) a local Ollama server by changing `base_url`.

**Next:** [Activity 2](./Activity_2_Tokens_and_Embeddings.ipynb) answers the question this notebook raised and skipped: what exactly is a token, and what is the other way text gets turned into numbers.